<a href="https://colab.research.google.com/github/vsoto89/Etapa_3_inte/blob/main/IEI095_Avance3_Limpieza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

archivos = [
    '/content/drive/MyDrive/dataset_ventas_1.xlsx',
    '/content/drive/MyDrive/dataset_ventas_2.xlsx',
    '/content/drive/MyDrive/dataset_ventas_3.xlsx'
]
print("Librerías importadas y rutas definidas.")

Librerías importadas y rutas definidas.


In [ ]:
# =====================================================================
# PIPELINE ETL: INGESTA, MERGE, LIMPIEZA TRANSACCIONAL Y DIAGNÓSTICO
# =====================================================================

lista_tablas_planas = []

# 1. Leemos y ensamblamos archivo por archivo
for ruta in archivos:
    v = pd.read_excel(ruta, sheet_name='Fact_Ventas')
    c = pd.read_excel(ruta, sheet_name='Dim_Clientes')
    p = pd.read_excel(ruta, sheet_name='Dim_Productos')
    s = pd.read_excel(ruta, sheet_name='Dim_Sucursales')

    # Adaptador de IDs y cruces
    tabla_plana = pd.merge(v, c, left_on='Cliente_ID', right_on='ID', how='left')
    tabla_plana = tabla_plana.drop(columns=['ID'])
    tabla_plana = pd.merge(tabla_plana, p, on='Producto_ID', how='left')
    tabla_plana = pd.merge(tabla_plana, s, on='Sucursal_ID', how='left')

    lista_tablas_planas.append(tabla_plana)

# 2. Apilamos todo en la placa base final (15.000 filas iniciales)
df_final = pd.concat(lista_tablas_planas, ignore_index=True)

# 3. LA REGLA DE VÍCTOR: LIMPIEZA DE TRANSACCIONES DUPLICADAS POR ERROR DE SISTEMA
# Filtramos directamente df_final comparando Nombre + Ciudad + Fecha + Producto
filas_antes = len(df_final)
df_final = df_final.drop_duplicates(subset=['Nombre_Completo', 'Ciudad_Base', 'Fecha', 'Producto_ID'])
filas_despues = len(df_final)
errores_eliminados = filas_antes - filas_despues

# 4. Estadísticas descriptivas (Contar personas reales)
clientes_unicos = df_final.drop_duplicates(subset=['Nombre_Completo', 'Ciudad_Base'])

print("¡Esquema Estrella unificado y limpieza transaccional aplicada!")
print("-" * 50)
print(f"Transacciones iniciales (con posibles errores): {filas_antes}")
print(f"Boletas duplicadas por error del sistema eliminadas: {errores_eliminados}")
print(f"Total Ventas Oficiales y Limpias: {filas_despues}")
print(f"Total Clientes Únicos Reales: {len(clientes_unicos)}")
print("-" * 50)
print("DIAGNÓSTICO DE CALIDAD DE DATOS (Nulos):")
print(df_final.isnull().sum())

¡Esquema Estrella unificado y limpieza transaccional aplicada!
--------------------------------------------------
Transacciones iniciales (con posibles errores): 15000
Boletas duplicadas por error del sistema eliminadas: 564
Total Ventas Oficiales y Limpias: 14436
Total Clientes Únicos Reales: 275
--------------------------------------------------
DIAGNÓSTICO DE CALIDAD DE DATOS (Nulos):
Venta_ID                        0
Fecha                           0
Cliente_ID                      0
Producto_ID                     0
Sucursal_ID                     0
Cantidad                        0
Precio_Unitario_CLP             0
Descuento_Aplicado_CLP          0
Monto_Total_CLP                 0
API_Clima_Temp_C                0
API_Clima_Condicion             0
API_Despacho_Entrega_Minutos    0
Nombre_Completo                 0
Genero                          0
Ciudad_Base                     0
Region_Base                     0
Nombre                          0
Categoría                      